In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
import joblib
import warnings
warnings.filterwarnings('ignore')

print("--- Starting Clean Model Training ---")

# 1. Load Data
try:
    data = pd.read_excel("Data_Train.xlsx")
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("ERROR: 'Data_Train.xlsx' not found. Please upload your dataset to this folder.")
    exit()

data.dropna(inplace=True)

# 2. Preprocessing Functions
def preprocess_duration(x):
    if 'h' not in x:
        x = '0h' + ' ' + x
    elif 'm' not in x:
        x = x + ' ' + '0m'
    return x

# 3. Apply Preprocessing
# Date_of_Journey
data["Journey_day"] = data['Date_of_Journey'].dt.day
data["Journey_month"] = data['Date_of_Journey'].dt.month

# Duration
data['Duration'] = data['Duration'].apply(preprocess_duration)
data['Duration_hours'] = data['Duration'].apply(lambda x : int(x.split(' ')[0][0:-1]))
data['Duration_mins'] = data['Duration'].apply(lambda x : int(x.split(' ')[1][0:-1]))

# Dep_Time
data["Dep_Time"] = pd.to_datetime(data["Dep_Time"])
data["Dep_Time_hour"] = data["Dep_Time"].dt.hour
data["Dep_Time_minute"] = data["Dep_Time"].dt.minute

# Arrival_Time
data["Arrival_Time"] = pd.to_datetime(data["Arrival_Time"])
data["Arrival_Time_hour"] = data["Arrival_Time"].dt.hour
data["Arrival_Time_minute"] = data["Arrival_Time"].dt.minute

# 4. ENCODING (The Fix: We Map Source, Destination, and Airline consistently)
print("Encoding categorical features...")

# Airline
airline_means = data.groupby(['Airline'])['Price'].mean().sort_values().index
dict_airlines = {key: index for index, key in enumerate(airline_means, 0)}
data['Airline'] = data['Airline'].map(dict_airlines)

# Destination (Fix New Delhi)
data['Destination'].replace('New Delhi', 'Delhi', inplace=True)
dest_means = data.groupby(['Destination'])['Price'].mean().sort_values().index
dict_dest = {key: index for index, key in enumerate(dest_means, 0)}
data['Destination'] = data['Destination'].map(dict_dest)

# Source (ADDED THIS to fix your error)
source_means = data.groupby(['Source'])['Price'].mean().sort_values().index
dict_source = {key: index for index, key in enumerate(source_means, 0)}
data['Source'] = data['Source'].map(dict_source)

# Total Stops
stop_map = {'non-stop': 0, '2 stops': 2, '1 stop': 1, '3 stops': 3, '4 stops': 4}
data['Total_Stops'] = data['Total_Stops'].map(stop_map)

# 5. Drop Unused Columns
cols_to_drop = ['Date_of_Journey', 'Additional_Info', 'Journey_year', 'Route', 'Duration', 'Dep_Time', 'Arrival_Time']
data.drop(columns=cols_to_drop, axis=1, inplace=True, errors='ignore')

# 6. Train Model
X = data.drop('Price', axis=1)
y = data['Price']

print(f"Training on features: {X.columns.tolist()}")
reg_rf = RandomForestRegressor(n_estimators=100, random_state=42)
reg_rf.fit(X, y)

# 7. Save Artifacts
print("Saving model and encoders...")
joblib.dump(reg_rf, 'Code_flight_rf_model.pkl')
joblib.dump({
    'airlines': dict_airlines,
    'destinations': dict_dest,
    'sources': dict_source,  # Saved Source Map
    'stops': stop_map,
    'columns': X.columns.tolist()
}, 'Code_encoding_maps.pkl')

print("SUCCESS! Model re-trained and saved. You can now run the App.")

--- Starting Clean Model Training ---
Dataset loaded successfully.


AttributeError: Can only use .dt accessor with datetimelike values